# Dynamic ACDC: MATLAB audit and initial MRzero comparison

This notebook reads the supplied results, checks the trajectory and constraints,
and compares the **same simplified physical model** in MATLAB, NumPy and MRzero.
The exact inputs for subject 006, MATLAB slice 120, are included in the local
archive. Their selection and units are documented in
`subject006_slice120_provenance.json`.

**Interpretation:** numerical agreement between simulators does not yet establish
intravoxel signal recovery or hardware feasibility. The original reconstruction
is preserved to reproduce the supplied results, with its grid defects explicitly
counted. This notebook does not rerun optimization. Fresh Python optimization is
available in `02_dynamic_shimming_MRzero.ipynb`.

Documents reviewed: the email “ACDC shim field optimization for EPI artifact reduction”,
chapter 6 *B0 inhomogeneity: Causes and coping strategies*, and the manuscript
*An open-source FPGA-based 64-channel arbitrary waveform controller...*.
API sources: [MRzero](https://github.com/MRsources/MRzero-Core),
[per-voxel magnetization](https://github.com/MRsources/MRzero-Core/blob/main/python/MRzeroCore/simulation/main_pass.py),
[sign conventions](https://mrsources.github.io/MRzero-Core/fft_sign_conventions.html).


In [ ]:
from pathlib import Path
from importlib.metadata import version
import json, sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from IPython.display import display, Markdown
import torch

HERE = Path.cwd()
if not (HERE / 'subject006_slice120_inputs.npz').exists():
    HERE = HERE / 'MRzero_validation'
ROOT = HERE.parent
sys.path.insert(0, str(HERE))
from mrzero_dynamic import simulate_dynamic_acdc, apply_dynamic_shim_phase, validate_synthetic
torch.set_num_threads(4)
versions = {p: version(p) for p in ['numpy','scipy','torch','MRzeroCore','matplotlib']}
print('Versions:', versions)
report = {'versions': versions, 'scope': 'single ideal RF, stationary point spins, original reconstruction'}

## 1. Fields, currents and coordinate frame

The code uses $f(\mathbf r,t)=f_{B0}(\mathbf r)+\sum_c b_c(\mathbf r)I_c(t)$,
with $b_c$ in Hz/A. The basis is precomputed from the coil wire paths using
Biot–Savart; `forward_model.m` receives only this basis and the X/Y coordinates.
Geometry and orientation are therefore **encoded in the maps**, assuming that
they share the subject coordinate frame. These exports contain no DICOM
orientation matrix or transformation for automatically changing slice orientation.
The maps below preserve the MATLAB array order `[x, y]` exactly.


In [ ]:
maps = dict(np.load(HERE / 'subject006_slice120_inputs.npz'))
base = loadmat(ROOT / 'sol_BASELINE.mat', simplify_cells=True)
params, seq = base['params'], base['seq']
assert np.array_equal(maps['X_m'], params['X'])
assert np.array_equal(maps['Y_m'], params['Y'])
provenance = json.loads((HERE / 'subject006_slice120_provenance.json').read_text())
report['input_selection'] = provenance['crop_MATLAB_inclusive_1based']
print('Inputs:', maps['img'].shape, 'channels:', maps['shim_Hz_per_A'].shape[-1])
print('Head/brain/constraint mask counts:', *(int(maps[k].sum()) for k in
      ['head_mask','brain_mask','constraint_mask']))
fig, axes = plt.subplots(1, 3, figsize=(12, 4), constrained_layout=True)
for ax, key, title in zip(axes, ['img','b0_Hz','constraint_mask'],
                         ['MPRAGE: initial amplitude', 'B0 [Hz]', 'Constraint mask']):
    im = ax.imshow(maps[key], cmap='gray' if key != 'b0_Hz' else 'coolwarm')
    ax.set_title(title); ax.axis('off'); fig.colorbar(im, ax=ax, shrink=.7)
plt.show()

## 2. ADC and reconstruction audit

The saved trajectory contains 32×32 ADC samples. Their coordinates lie at
half-integers rather than on the integer grid expected by the centered FFT.
Negative readout lines also have an offset. MATLAB `round` followed by the
boundary check does not perform physical resampling: some ADC samples are
discarded and one grid cell can be overwritten. Check the trajectory before
optimizing a corrected physical model; agreement between software implementations
can reproduce the same defect. The fresh Python notebook intentionally preserves
this reconstruction when comparing the original method.


In [ ]:
N, dt = int(params['matrix_size']), float(seq['dt'])
adc_steps = np.flatnonzero(np.asarray(seq['adc'], dtype=bool))
scaled = np.stack([seq['kx_adc'], seq['ky_adc']], axis=-1) / seq['dk']
# MATLAB round: half-integers are rounded away from zero.
bins = (np.sign(scaled)*np.floor(np.abs(scaled)+.5)).astype(int) + N//2
valid = np.all((bins >= 0) & (bins < N), axis=1)
unique_bins = np.unique(bins[valid], axis=0).shape[0]
trajectory = dict(adc_count=int(len(adc_steps)), kept=int(valid.sum()),
                  rejected=int((~valid).sum()), unique_bins=int(unique_bins),
                  duration_ms=float(len(seq['Gx'])*dt*1000),
                  echo_spacing_ms=float(np.median(np.diff(adc_steps[::N]))*dt*1000))
report['trajectory'] = trajectory
print(json.dumps(trajectory, indent=2))
assert trajectory['adc_count'] == 1024 and trajectory['kept'] == 960
assert trajectory['unique_bins'] == 959
fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)
ax.plot(scaled[:,0], scaled[:,1], '.', ms=2, label='Saved ADC samples')
ax.plot(scaled[~valid,0], scaled[~valid,1], 'rx', label='Discarded ADC samples')
ax.set(xlabel='kx / dk', ylabel='ky / dk', title='Trajectory used in the saved MATLAB results')
ax.legend(); ax.set_aspect('equal'); plt.show()

def reconstruct_original(signal, bins=bins, valid=valid, N=N):
    kdata = np.zeros((N, N), dtype=complex)
    # Explicitly keep the last ADC sample when grid cells collide.
    for j in np.flatnonzero(valid):
        kdata[tuple(bins[j])] = signal[j]
    image = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(kdata)))
    return image, kdata

def nrmse_magnitude(image, reference):
    return float(100*np.linalg.norm(np.abs(image)-np.abs(reference))/np.linalg.norm(reference))

def relative_complex(actual, reference):
    return float(np.linalg.norm(actual-reference)/np.linalg.norm(reference))

## 3. Saved results and actual constraints

Conventional shimming minimizes the residual slice B0 and keeps the currents
constant during the train. Mode 3 optimizes the currents for each line to minimize
**image magnitude error**, giving 32×32 unknowns.
Field constraints apply only to the 270 points of the sparse mask.
The “full head” column measures the maximum over all 2498 head points.
These limits concern the added ACDC field, rather than the total residual field.


In [ ]:
solutions = {}
rows = []
for limit in [25, 50, 100, 200, 300, 500, 1000]:
    d = loadmat(ROOT / f'sol_shimOpt3_ACDCFieldLimit{limit}.mat',
                variable_names=['Ishim2','img_acdc2','params','seq'], simplify_cells=True)
    solutions[limit] = d
    currents = d['Ishim2']
    fields = maps['shim_Hz_per_A'].reshape(-1, 32) @ currents
    row = dict(nominal_limit_Hz=limit,
               nrmse_percent=nrmse_magnitude(d['img_acdc2'], base['img_ref']),
               max_current_A=float(np.max(np.abs(currents))),
               max_delta_I_A=float(np.max(np.abs(np.diff(currents,axis=1)))),
               max_sum_abs_current_A=float(np.max(np.sum(np.abs(currents),axis=0))),
               sparse_field_Hz=float(np.max(np.abs(fields[maps['constraint_mask'].ravel()]))),
               full_head_field_Hz=float(np.max(np.abs(fields[maps['head_mask'].ravel()]))))
    rows.append(row)
table = '| Nominal limit Hz | NRMSE % | max current A | max ΔI A | sparse-mask field Hz | full-head field Hz |\n'
table += '|---:|---:|---:|---:|---:|---:|\n'
for r in rows:
    table += '| {nominal_limit_Hz} | {nrmse_percent:.2f} | {max_current_A:.2f} | {max_delta_I_A:.2f} | {sparse_field_Hz:.1f} | {full_head_field_Hz:.1f} |\n'.format(**r)
display(Markdown(table))
report['saved_dynamic_results'] = rows
report['saved_baseline_nrmse_percent'] = {k:nrmse_magnitude(base[k],base['img_ref'])
                                        for k in ['img_2sh','img_acdc']}
print('Baseline errors:', report['saved_baseline_nrmse_percent'])
fig, axes = plt.subplots(1, 5, figsize=(15, 4), constrained_layout=True)
images = [base['img_ref'], base['img_2sh'], base['img_acdc'],
          solutions[100]['img_acdc2'], solutions[1000]['img_acdc2']]
for ax, image, title in zip(axes, images,
            ['B0=0','2SH','Conventional ACDC','Mode 3, 100 Hz limit','Mode 3, 1000 Hz limit']):
    ax.imshow(np.abs(image), cmap='gray', vmin=0, vmax=np.max(np.abs(base['img_ref'])))
    ax.set_title(title, fontsize=10); ax.axis('off')
fig.savefig(HERE / 'saved_results_comparison.png', dpi=160)
plt.show()

## 4. MRzero: ACDC phase before summing voxels

MRzero 1.1.1 uses a static B0. For this GRE-EPI sequence with **one ideal RF pulse**,
`execute_graph(return_mag_adc=True)` exposes the transverse magnetization per voxel.
The adapter applies to each voxel
$\exp[-2\pi i\sum_c b_c(\mathbf r)\int_0^t I_c(u)\,du]$,
then sums the receive signal. It also integrates transitions without ADC samples.
This operation is exact for stationary point spins after the single RF pulse;
it cannot be applied to a signal that has already been summed.

`mrzero_dynamic.py` contains the full implementation. It uses a phantom with
explicit positions and unnormalized gradients to preserve X/Y, the origin shift
and the MATLAB gamma. T2′, diffusion and voxel-shape attenuation are disabled.
Native calculations use float32; the NumPy reference uses float64.
Multiple RF pulses, a realistic slice profile or subvoxel field gradients require
an extension of Bloch/PDG propagation and a convergence study.


In [ ]:
synthetic_checks = validate_synthetic()
report['synthetic_signal_checks'] = synthetic_checks
print(json.dumps(synthetic_checks, indent=2))

# Stored references previously computed using matlab -batch and forward_model.m.
case = loadmat(HERE / 'synthetic_matlab_input.mat', simplify_cells=True)
matlab_case = loadmat(HERE / 'synthetic_matlab_output.mat', simplify_cells=True)
result = simulate_dynamic_acdc(case['img'],case['b0'],case['acdc_maps'],
                               case['Ishim'],case['params'],case['seq'])
matlab_bins = np.column_stack([matlab_case['ix_matlab'],matlab_case['iy_matlab']]).astype(int)-1
matlab_valid = np.asarray(matlab_case['valid_matlab'],dtype=bool)
case_image, case_k = reconstruct_original(result['signal_adc'],matlab_bins,matlab_valid)
static_image, static_k = reconstruct_original(result['static_signal_adc'],matlab_bins,matlab_valid)
case_errors = dict(dynamic_image=relative_complex(case_image,matlab_case['img_matlab']),
                   static_image=relative_complex(static_image,matlab_case['img_static_matlab']),
                   dynamic_signal_numpy=result['relative_error'])
report['executed_matlab_synthetic_comparison'] = case_errors
print('Complex relative errors against MATLAB:', case_errors)
assert max(case_errors.values()) < 5e-5

## 5. Comparison using the original subject 006 inputs

First compare the saved optimized currents using the same trajectory, units and
reconstruction. References are not renormalized to hide a scale or phase mismatch.
The observed agreement validates the point-spin model port; it does not correct
the trajectory defects identified above.


In [ ]:
real_cases = {
    'B0_zero': (np.zeros_like(maps['b0_Hz']), np.zeros_like(base['Ishim']), base['img_ref'],params,seq),
    '2SH': (maps['b0_Hz'], np.zeros_like(base['Ishim']), base['img_2sh'],params,seq),
    'conventional_ACDC': (maps['b0_Hz'],base['Ishim'],base['img_acdc'],params,seq),
    'dynamic_100Hz': (maps['b0_Hz'],solutions[100]['Ishim2'],solutions[100]['img_acdc2'],solutions[100]['params'],solutions[100]['seq']),
    'dynamic_1000Hz': (maps['b0_Hz'],solutions[1000]['Ishim2'],solutions[1000]['img_acdc2'],solutions[1000]['params'],solutions[1000]['seq']),
}
real_errors = {}
for name,(b0,currents,reference,case_params,case_seq) in real_cases.items():
    assert np.array_equal(case_seq['kx_adc'],seq['kx_adc'])
    assert np.array_equal(case_seq['ky_adc'],seq['ky_adc'])
    result = simulate_dynamic_acdc(maps['img'],b0,maps['shim_Hz_per_A'],currents,case_params,case_seq,
                                   return_native_magnetization=(name=='dynamic_1000Hz'))
    mrzero_image,_ = reconstruct_original(result['signal_adc'])
    numpy_image,_ = reconstruct_original(result['analytical_signal_adc'])
    errors = dict(mrzero_vs_saved_matlab=relative_complex(mrzero_image,reference),
                  numpy_vs_saved_matlab=relative_complex(numpy_image,reference),
                  mrzero_vs_numpy_signal=result['relative_error'])
    assert errors['mrzero_vs_saved_matlab'] < 5e-5, (name,errors)
    assert errors['numpy_vs_saved_matlab'] < 1e-10, (name,errors)
    real_errors[name] = errors
    print(name, errors)
report['subject006_comparison'] = real_errors
report['gamma_Hz_per_T_by_case'] = {name:float(entry[3]['gammaHz']) for name,entry in real_cases.items()}
print('Gamma in saved files:',report['gamma_Hz_per_T_by_case'])

## 6. Illustrative sensitivity to current response

The filtered 1 A step described in the manuscript reaches 99% in 133 µs.
A first-order model gives $\tau=133\,\mu s/\ln(100)\simeq29\,\mu s$,
consistent with a 5.5 kHz bandwidth. This model remains **an illustrative
assumption**: the response of each channel must be measured with the coil and
filters actually used.
The currents below are exact averages over each 8 µs interval so that their phase
is integrated correctly. Clipping at ±3 A is a sensitivity study of the saved
solutions; optimization under these constraints would be needed to assess the
achievable performance.


In [ ]:
I = solutions[1000]['Ishim2']
events = np.asarray(seq['eventOfTime'],dtype=int)
command = np.zeros((I.shape[0],len(events)))
active = events>0
command[:,active] = I[:,events[active]-1]
tau = 133e-6/np.log(100)
def first_order_interval_average(command,tau,dt):
    alpha = np.exp(-dt/tau)
    state = np.zeros(command.shape[0])
    average = np.zeros_like(command)
    for t in range(command.shape[1]):
        target=command[:,t]
        average[:,t]=target+(state-target)*(tau/dt)*(1-alpha)
        state=target+(state-target)*alpha
    return average
filtered = first_order_interval_average(command,tau,dt)
clipped_filtered = first_order_interval_average(np.clip(command,-3,3),tau,dt)
hardware_sensitivity = {}
for label,waveform in [('ideal_5A',command),('first_order_5A',filtered),
                       ('clip3A_first_order',clipped_filtered)]:
    signal=apply_dynamic_shim_phase(result['voxel_magnetization_adc'],maps['shim_Hz_per_A'],I,seq,
          currents_by_time_a=waveform,receive_sensitivity=result['receive_sensitivity'])
    image,_=reconstruct_original(signal)
    hardware_sensitivity[label]=nrmse_magnitude(image,base['img_ref'])
print('Magnitude NRMSE, sensitivity of saved currents (%):',hardware_sensitivity)
report['illustrative_first_order_response']={'tau_s':float(tau),'nrmse_percent':hardware_sensitivity,
       'status':'illustrative assumed response; clipped currents have not been reoptimized'}
c=int(np.argmax(np.max(np.abs(np.diff(I,axis=1)),axis=1)))
time_ms=(np.arange(len(events))+.5)*dt*1000
fig,ax=plt.subplots(figsize=(10,3),constrained_layout=True)
for label,waveform in [('Command',command),('Assumed response',filtered),
                       ('3 A clipping + response',clipped_filtered)]:
    ax.plot(time_ms,waveform[c],label=label)
ax.set(xlabel='Time after RF [ms]',ylabel='Current [A]',title=f'Example: MATLAB channel {c+1}')
ax.legend(); plt.show()

## 7. Transfer to other subjects

The `eval_other_subj` files apply the subject 006 currents to other subjects;
they do not represent complete individual optimizations.
Large subject-specific fields can therefore worsen other subjects' images.


In [ ]:
transfer_rows=[]
table='| Subject | Conventional ACDC % | Transfer 100 Hz % | Transfer 1000 Hz % |\n|---|---:|---:|---:|\n'
for subject in ['007','008','009']:
    directory=ROOT/'eval_other_subj'/subject
    b=loadmat(directory/'sol_BASELINE.mat',variable_names=['img_ref','img_acdc'],simplify_cells=True)
    vals=[nrmse_magnitude(b['img_acdc'],b['img_ref'])]
    for limit in [100,1000]:
        d=loadmat(directory/f'sol_shimOpt3_ACDCFieldLimit{limit}.mat',variable_names=['img_acdc2'],simplify_cells=True)
        vals.append(nrmse_magnitude(d['img_acdc2'],b['img_ref']))
    transfer_rows.append(dict(subject=subject,conventional_percent=vals[0],transfer100_percent=vals[1],transfer1000_percent=vals[2]))
    table += f'| {subject} | {vals[0]:.2f} | {vals[1]:.2f} | {vals[2]:.2f} |\n'
display(Markdown(table)); report['transferred_results']=transfer_rows

## 8. Additional validation for publication

1. Correct ADC positions and verify Cartesian coverage of 1024 unique cells,
   or reconstruct the actual trajectory without rounding. Also unify the three
   gamma constants used in the original code.
2. Rerun optimization with the full constraint mask, current limits compatible
   with the actual system, and an identified temporal response. The Rev. D
   manuscript recommends ±3 A per channel and measures 133 µs to 99% for a
   filtered 1 A step. The 8 µs period describes digital update timing, rather
   than settling time.
3. Add a 3D volume covering the slice thickness, subvoxels and selective RF to
   measure signal recovery, with resolution convergence. The current model has
   one B0 value per point; summing can dephase neighboring points, but does not
   describe field variation within their volume.
4. Compare multiple subjects and slices, including anatomies and contrasts not
   used during optimization; then accelerated EPI with matched receive
   sensitivities and reconstruction. The email already raises these questions.
5. Add a B0-aware reconstruction comparator to distinguish prospective gains
   from improvements achievable retrospectively.

Other code findings: mode 2 has inconsistent dimensions and constraints, and the
declared function name `epi_acdc_artCorr` differs from `epi_acdc_artCor.m`. Saved
mode 1 solutions need transposition to follow the current `[channel, line]`
convention. These modes are not used in the comparison above.

A phase correction computed in MRzero with the same assumptions does not, by
itself, independently validate all the physics. Further work should compare
against a Bloch/isochromat simulator with multiple spins per voxel and the actual
current response. The fresh Python notebook retains the original model and
constraints to keep its comparison aligned with the supplied method.


In [ ]:
(HERE/'comparison_report.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
print('Report:',HERE/'comparison_report.json')